In [ ]:
import jax.numpy as jnp
from typing import Callable

def conjugate_gradient(
    A: Callable[[jnp.ndarray], jnp.ndarray], 
    b: jnp.ndarray, 
    x0: jnp.ndarray, 
    tol: float = 1e-8, 
    max_iter: int = 1000
) -> jnp.ndarray:
    """
    Solves Ax = b using the Conjugate Gradient method.
    Note: jnp.dot has been replaced with jnp.vdot to correctly handle 
    multi-dimensional array inner products.
    """
    x = jnp.copy(x0)
    r = b - A(x)
    p = jnp.copy(r)
    last_res = jnp.vdot(r, r)

    for _ in range(max_iter):
        Ap = A(p)
        pAp = jnp.vdot(p, Ap)

        alpha = jnp.where(pAp == 0.0, 0.0, last_res / pAp)
        x = x + alpha * p
        r = r - alpha * Ap
        res = jnp.vdot(r, r)

        if res < tol**2:
            break

        beta = res / last_res
        p = r + beta * p
        last_res = res

    return x


def solve_poisson_cylindrical(
    f: jnp.ndarray,
    r_1D: jnp.ndarray,
    theta_1D: jnp.ndarray,
    z_1D: jnp.ndarray,
    x0: jnp.ndarray | None = None,
    tol: float = 1e-8,
    max_iter: int = 1000
) -> jnp.ndarray:
    """
    Implicitly solves ∇²φ = f on a cylindrical grid with vanishing Dirichlet 
    boundary conditions in r and z, and periodic boundary conditions in θ.
    
    Parameters
    ----------
    f : jnp.ndarray
        Source term array of shape (N_r-2, N_θ, N_z-2) defined on interior points.
    r_1D, theta_1D, z_1D : jnp.ndarray
        1D coordinate arrays including boundary points.
    x0 : jnp.ndarray, optional
        Initial guess. Defaults to zero array.
    tol, max_iter : float, int
        Solver tolerance and iteration limit.
        
    Returns
    -------
    jnp.ndarray
        Solution φ on the interior grid with shape (N_r-2, N_θ, N_z-2).
    """
    # Extract interior coordinates and uniform spacings
    r_int = r_1D[1:-1]
    dr = r_1D[1] - r_1D[0]
    dtheta = theta_1D[1] - theta_1D[0]
    dz = z_1D[1] - z_1D[0]
    
    # Reshape radial coordinate for explicit broadcasting over θ and z
    r_int_3d = r_int[:, jnp.newaxis, jnp.newaxis]
    
    def laplacian(phi: jnp.ndarray) -> jnp.ndarray:
        # Zero-padding enforces vanishing Dirichlet BCs at r_min, r_max, z_min, z_max
        phi_padded = jnp.pad(phi, ((1, 1), (0, 0), (1, 1)), constant_values=0.0)
        
        phi_rp = phi_padded[2:, :, :]
        phi_rm = phi_padded[:-2, :, :]
        phi_rc = phi  # Equivalent to phi_padded[1:-1, :, :]
        
        # Periodic handling in θ via circular shift
        phi_tp = jnp.roll(phi, -1, axis=1)
        phi_tm = jnp.roll(phi,  1, axis=1)
        
        phi_zp = phi_padded[:, :, 2:]
        phi_zm = phi_padded[:, :, :-2]
        
        # Discrete cylindrical Laplacian: ∂²/∂r² + (1/r)∂/∂r + (1/r²)∂²/∂θ² + ∂²/∂z²
        return (
            (phi_rp - 2.0 * phi_rc + phi_rm) / dr**2 +
            (phi_rp - phi_rm) / (2.0 * dr * r_int_3d) +
            (phi_tp - 2.0 * phi_rc + phi_tm) / (dtheta**2 * r_int_3d**2) +
            (phi_zp - 2.0 * phi_rc + phi_zm) / dz**2
        )

    # The discrete Laplacian operator is negative-definite. Conjugate Gradient 
    # requires a positive-definite system. We solve -∇²φ = -f instead.
    A_op = lambda phi: -laplacian(phi)
    b_vec = -f
    
    if x0 is None:
        x0 = jnp.zeros_like(f)
        
    return conjugate_gradient(A_op, b_vec, x0, tol=tol, max_iter=max_iter)

# Define 1D grids
N_r, N_theta, N_z = 32, 64, 32
r_min, r_max = 0.0, 5.0
z_min, z_max = -5.0, 5.0

r_1D = jnp.linspace(r_min, r_max, N_r)
theta_1D = jnp.linspace(0, 2*jnp.pi, N_theta)
z_1D = jnp.linspace(z_min, z_max, N_z)

# Create interior source term (e.g., Gaussian charge distribution)
r_int = r_1D[1:-1]
z_int = z_1D[1:-1]
R, _, Z = jnp.meshgrid(r_int, theta_1D, z_int, indexing='ij')
f_source = jnp.exp(-(R**2 + Z**2))

# Solve Poisson's equation
phi_solution = solve_poisson_cylindrical(f_source, r_1D, theta_1D, z_1D, tol=1e-6)